<h3> Projekt Nauczanie Maszynowe </h3>

<h4> 1. Biblioteki

In [3]:
import numpy as np
import pandas as pd
from scipy.io import mmread
import torch
import matplotlib as mt

<h4> 2. Dane

In [6]:
genes = pd.read_csv("genes.tsv", sep="\t", header=None)[0].values

In [8]:
barcodes = pd.read_csv("barcodes.tsv", sep="\t", header=None)[0].values

In [14]:
meta = pd.read_csv("SCP_metadata_20261216.txt", sep="\t", skiprows=[1], index_col=0)

In [16]:
valid_meta = meta[(meta['percent_mt'] < 10) & (meta['nFeature_RNA'] > 200)]
valid_barcodes = valid_meta.index.intersection(barcodes)

In [18]:
barcode_to_idx = {bc: i for i, bc in enumerate(barcodes)}
valid_indices = [barcode_to_idx[bc] for bc in valid_barcodes]

In [20]:
matrix = mmread("matrix.mtx").T 
matrix_filtered = matrix.tocsr()[valid_indices, :]

In [21]:
df = pd.DataFrame(matrix_filtered.toarray(), index=valid_barcodes, columns=genes)

In [22]:
df = df.join(valid_meta[['percent_mt', 'nFeature_RNA', 'leukemia', 'SingleR_hpca']])

In [28]:
geny_kolumny = [c for c in df.columns if c not in ['percent_mt', 'nFeature_RNA', 'leukemia', 'SingleR_hpca']]

In [30]:
#Normalizacja

In [31]:
sumy_komorek = df[geny_kolumny].sum(axis=1)

In [ ]:
df[geny_kolumny] = df[geny_kolumny].div(sumy_komorek, axis=0) * 10000

In [90]:
#Logarytmowanie

In [92]:
#Skalowanie

In [45]:
# Tworzymy funkcję decyzyjną
def przypisz_etykiete(row):
    # Jeśli to niedojrzała komórka B u pacjenta z białaczką B-ALL
    if row['leukemia'] == 'B-ALL' and 'Pro-B' in str(row['SingleR_hpca']):
        return 1  # 1 = Blast (Chora)
    
    # Jeśli to dojrzała komórka odpornościowa (T, NK lub Monocyt)
    if row['SingleR_hpca'] in ['T_cells', 'Monocyte', 'NK_cell']:
        return 0  # 0 = Zdrowa
        
    return -1 # Reszta (do odrzucenia, żeby nie mylić modelu)

# Nakładamy funkcję na metadane
meta_data['label'] = meta_data.apply(przypisz_etykiete, axis=1)

# Sprawdzamy, ile mamy teraz przykładów do nauki:
print(meta_data['label'].value_counts())

label
 1    83237
 0    39526
-1    21984
Name: count, dtype: int64
